# ⚔️ CRUSADER — F01 GRIMALDUS
## Transcription Audio → timing.json

> *"The Chaplain speaks, and his words are the Emperor's truth."*

---

**Ce notebook doit tourner avec Runtime → GPU (T4)**

### Étapes :
1. Vérification GPU
2. Montage Google Drive
3. Installation faster-whisper
4. Téléchargement du script depuis GitHub
5. Configuration des chemins
6. Validation CUSTOS check-out
7. Lancement de la transcription
8. Validation CUSTOS check-in
9. Aperçu du timing.json produit

---
## Étape 1 — Vérification GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'[OK] GPU détecté : {result.stdout.strip()}')
else:
    print('[ATTENTION] Aucun GPU détecté. Allez dans Runtime → Changer le type de runtime → GPU (T4)')
    print('La transcription fonctionnera mais sera BEAUCOUP plus lente.')

---
## Étape 2 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 3 — Installation faster-whisper

In [ ]:
import subprocess, sys

print('Installation de faster-whisper...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'faster-whisper>=1.0.0', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    from faster_whisper import WhisperModel
    print('[OK] faster-whisper installé et importé avec succès.')
else:
    print('[ERREUR]', result.stderr)

---
## Étape 4 — Téléchargement du script depuis GitHub

In [ ]:
import urllib.request, os

SCRIPT_URL = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main/F01_GRIMALDUS/F01B_GRIMALDUS/CODEBASE/crs_f01_grimaldus.py'
CUSTOS_URL = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main/CRS_CUSTOS.py'

os.makedirs('/content/crusader', exist_ok=True)

urllib.request.urlretrieve(SCRIPT_URL, '/content/crusader/crs_f01_grimaldus.py')
print('[OK] crs_f01_grimaldus.py téléchargé')

urllib.request.urlretrieve(CUSTOS_URL, '/content/crusader/CRS_CUSTOS.py')
print('[OK] CRS_CUSTOS.py téléchargé')

---
## Étape 5 — Configuration des chemins

**Seule cellule à modifier.** Adaptez `CAMPAIGN_NAME` si vous travaillez sur une campagne différente.

In [ ]:
import os

# ════════════════════════════════════════
#  CONFIGURATION — À ADAPTER SI BESOIN
# ════════════════════════════════════════
DRIVE_BASE    = '/content/drive/MyDrive/DRIVE_CRUSADER'
FPS           = 30
WHISPER_MODEL = 'medium'   # tiny | base | small | medium | large-v3
# ════════════════════════════════════════

PATH_IN  = os.path.join(DRIVE_BASE, 'F01_GRIMALDUS', 'IN')
PATH_OUT = os.path.join(DRIVE_BASE, 'F01_GRIMALDUS', 'OUT')

os.makedirs(PATH_IN,  exist_ok=True)
os.makedirs(PATH_OUT, exist_ok=True)

print(f'IN  → {PATH_IN}')
print(f'OUT → {PATH_OUT}')
print(f'FPS : {FPS} | Modèle Whisper : {WHISPER_MODEL}')

audio_file = os.path.join(PATH_IN, 'audio_clean.mp3')
if os.path.isfile(audio_file):
    size_mb = os.path.getsize(audio_file) / 1024 / 1024
    print(f'[OK] audio_clean.mp3 présent ({size_mb:.2f} MB)')
else:
    print(f'[ATTENTION] audio_clean.mp3 ABSENT de {PATH_IN}')
    print('  → Copiez votre fichier audio dans ce dossier Drive avant de continuer.')

---
## Étape 6 — Validation CUSTOS check-out

In [ ]:
import subprocess

result = subprocess.run(
    ['python', '/content/crusader/CRS_CUSTOS.py',
     '--frigate', 'F01',
     '--mode', 'check-out',
     '--drive-base', DRIVE_BASE],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[BLOQUÉ] CUSTOS check-out échoué. Corrigez les erreurs avant de continuer.')
    raise SystemExit(1)
else:
    print('[CUSTOS] Check-out validé — Lancement autorisé.')

---
## Étape 7 — Lancement de la transcription

In [ ]:
import subprocess

result = subprocess.run(
    ['python', '/content/crusader/crs_f01_grimaldus.py',
     '--input',  PATH_IN,
     '--output', PATH_OUT,
     '--fps',    str(FPS),
     '--model',  WHISPER_MODEL],
    capture_output=False   # Affichage en temps réel
)

if result.returncode != 0:
    print('[ERREUR] La transcription a échoué.')
    raise SystemExit(1)

---
## Étape 8 — Validation CUSTOS check-in

In [ ]:
import subprocess

result = subprocess.run(
    ['python', '/content/crusader/CRS_CUSTOS.py',
     '--frigate', 'F01',
     '--mode', 'check-in',
     '--drive-base', DRIVE_BASE],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[BLOQUÉ] CUSTOS check-in échoué. Le timing.json est manquant ou invalide.')
    raise SystemExit(1)
else:
    print('[CUSTOS] Check-in validé — F01 GRIMALDUS SCELLÉE.')

---
## Étape 9 — Aperçu du timing.json produit

In [ ]:
import json, os

timing_path = os.path.join(PATH_OUT, 'timing.json')

with open(timing_path, 'r', encoding='utf-8') as f:
    timing = json.load(f)

meta = timing['meta']
print('═══════════════════════════════════════')
print('  RAPPORT F01 GRIMALDUS')
print('═══════════════════════════════════════')
print(f"  Langue       : {meta['language']} ({meta['language_probability']:.2%})")
print(f"  Durée        : {meta['duration_seconds']}s = {meta['total_frames']} frames")
print(f"  FPS          : {meta['fps']}")
print(f"  Modèle       : {meta['model']}")
print(f"  Mots         : {meta['word_count']}")
print(f"  Mots forts   : {meta['strong_word_count']}")
print(f"  Segments     : {len(timing['segments'])}")
print('═══════════════════════════════════════')
print()
print('  Premiers mots transcrits :')
for w in timing['words'][:8]:
    strong_tag = ' ★' if w['is_strong'] else ''
    print(f"    [{w['start_frame']:>4}f → {w['end_frame']:>4}f] '{w['word']}'{strong_tag}")
print()
print('  Segments :')
for s in timing['segments']:
    print(f"    [{s['start_frame']:>4}f → {s['end_frame']:>4}f] {s['text']}")